# B2b: Representations trained with labels

TAR (E5-small-v2 LoRA per outer fold, per-column PCA32 or joint PCA64) and a label-trained classifier: L2 logistic regression on the full standardized frozen embedding of each encoder, per text column or joint, with output width equal to the decision features (regression: nine equal-frequency training bins); C from five fixed values by inner CV on training rows, five-part cross-fitting for training rows and a full-training fit for test rows; missing text stays missing. Row-level complementarity compares the rows each arm helps. Common protocol: seven fixed predictors with library defaults and no tuning (linear/logistic, CatBoost, LightGBM, TabPFN-3.5, TabICL v2, LimiX-2, Kumo); the baseline uses the non-text columns only; 18 datasets with the same five outer folds. Fixed headline metrics: binary ROC-AUC, multiclass macro one-vs-rest ROC-AUC, regression R². Datasets are the unit: per-dataset paired gain, equal-dataset means with dataset-bootstrap intervals, win/tie/loss counts, raw gain and share of headroom. 

Tables come from the B2 export, `output_JEVPFN/exp05_frozen_embeddings/results/group_b/b2_representations.zip`.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib.pyplot as plt
from src.utils.notebook_display import display_local, display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import benchmark_export, study_export
from src.visualize import figures, group_b as gbfig, style
style.apply()
NOTEBOOK = "exp06_tar"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope, provenance and the export read by this notebook")
display_json("Notebook execution identity", provenance())
frames, receipt = benchmark_export.read("B2")
print(study_export.describe(receipt))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
WAITING = "Waiting for export: no measurements are shown."


Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\exp05_frozen_embeddings\results\group_b\b2_representations.zip is absent. No measurements are shown.


## Coverage

In [2]:
report_section("2. Coverage: planned and complete arms")
if not frames:
    print(WAITING)
else:
    display_table("Planned versus complete five-fold arms per phase and predictor", frames["coverage"], preview_rows=14)
    record(frames["coverage"], "Coverage")


Waiting for export: no measurements are shown.


## TAR and the label-trained classifier

In [3]:
report_section("3. TAR and the label-trained classifier")
if not frames:
    print(WAITING)
else:
    summary = benchmark_export.median_summary(frames)
    rows = sorted(r for r in summary.representation.unique() if r.startswith(("tar/", "probe/")))
    rows += ["decision/both/median"]
    table = summary[summary.representation.isin(rows)]
    display_table("Equal-dataset gains of label-trained arms and the median decision model", table, preview_rows=21)
    record(table, "B2b summary")
    gbfig.render(save, gbfig.gain_forest(table, rows, name="b2b_gain", what="Representations trained with labels"))
    contrasts = frames["contrasts"]
    contrasts = contrasts[contrasts.family.isin(["tar", "probe"])]
    display_table("Median decision model (both) minus each label-trained arm", contrasts, preview_rows=12)
    record(contrasts, "B2b contrasts per dataset")


Waiting for export: no measurements are shown.


## Row-level complementarity

Per dataset and predictor: Spearman correlation of per-row gains between each decision arm and each embedding/TAR/classifier arm on identical rows and folds, and the overlap of their 10% most-improved rows against its chance level.

In [4]:
report_section("4. Row-level complementarity")
if not frames or frames["complementarity_summary"].empty:
    print(WAITING if not frames else "Row-level complementarity summaries are pending in this export.")
else:
    display_table("Equal-dataset complementarity summary", frames["complementarity_summary"], preview_rows=12)
    record(frames["complementarity_summary"], "Complementarity summary")


Waiting for export: no measurements are shown.


## Complete report

In [5]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified export was read; no inference, model training or live VSC changes were performed.")

exp06_tar

1. Scope, provenance and the export read by this notebook


Notebook execution identity
{
  "code_sha256": "6fefc8d71fb8d4ede1d1459074ab16e1630d80996c3504f09c91cc61f2f6d4c7",
  "git_head": "afff4cf5f464ef78b0e6d086b12e595a153fe72f",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\exp05_frozen_embeddings\\results\\group_b\\b2_representations.zip",
  "created_utc": null,
  "origin": null,
  "sha256": null,
  "bytes": null,
  "schema": null,
  "rece